**Fly Ergo Fly**

En esta sección importaremos todas las librerías con las que trabajaremos

In [17]:
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
from scipy import signal
from scipy.signal import find_peaks
from scipy.optimize import curve_fit

En esta sección definimos las rutas desde las que cargaremos nuestros electrorretinogramas (input) y la carpeta de salida (output) donde queremos que se guarden nuestros resultados. Es importante que en la carpeta input solo estén presentes los electrorretinogramas

In [19]:
input_dir = '/home/diego/BigData/ERG/Todos/'
output = '/home/diego/BigData/ERG/output/'

os.chdir(input_dir) 
names = []
pos = 0
for root, dirs, files in os.walk("."):
    for filename in files:
        names.append(filename)
        print("posicion %d: %s" %(pos, names[pos]))
        pos = pos + 1

posicion 0: ensayos th nuevos000440.dat_.txt
posicion 1: th-synf 40 días000624.dat_.txt
posicion 2: th-synf 40 días000559.dat_.txt
posicion 3: ensayos th nuevos000386.dat_.txt
posicion 4: th-synf 40 días000601.dat_.txt
posicion 5: ensayos th nuevos000431.dat_.txt
posicion 6: th-synf 40 días nicotina000642.dat_.txt
posicion 7: th-synf 40 días000631.dat_.txt
posicion 8: ensayos th nuevos000447.dat_.txt
posicion 9: th-synf 40 días nicotina000651.dat_.txt
posicion 10: ensayos th nuevos000450.dat_.txt
posicion 11: ensayos th nuevos000399.dat_.txt
posicion 12: ensayos th nuevos000435.dat_.txt
posicion 13: th-synf 40 días nicotina000646.dat_.txt
posicion 14: ensayos th nuevos000394.dat_.txt
posicion 15: ensayos th nuevos000426.dat_.txt
posicion 16: th-synf 40 días nicotina000641.dat_.txt
posicion 17: th-synf 40 días000572.dat_.txt
posicion 18: th-synf 40 días000550.dat_.txt
posicion 19: ensayos th nuevos000418.dat_.txt
posicion 20: ensayos th nuevos000405.dat_.txt
posicion 21: th-synf 40 días

En esta parte calculamos los parámetros de nuestros electrorretinogramas. 
Dependiendo del ruido de los registros y las condiciones, algunos parámetros
han de ser modificados

In [20]:
def func_pol_1(x, a, b):
    return (a*x + b)

def func_pol_3(x, a, b, c, d):
    return (a*x**3 + b*x**2 + c*x + d)

def func_pol_4(x, a, b, c, d, e):
    return (a*x**4 + b*x**3 + c*x**2 + d*x + e)

resultados_erg = []
values = []
t_off = np.empty(0)
t_off_2 = np.empty(0)
B_off_2 = []

h_on = 0 #############################(Límite on)###################################################
range_on = 100 #########################(rango dcha on)###################################################
limit_counter_on = 50
t_ini_on = 0

h_off = 0.0 ############################(Límite off)#################################################
range_off = 900 ####################(rango dcha búsqueda off)#####################################
range_off_mins = 50 #######################(rango mínimos off 50)##########################################
range_off_2 = 50 ####################(rango izqda búsqueda off 2)#####################################
min_off = 0.2 ##############(mínimo valor de off para que sea aceptado)##############################

h_AmpR = 1  #############################(height peaks AmpR)###################################################
l_AmpR = 180 ############################(length line plot OvS)############################################

h_OvS = 1  #############################(height peaks OvS)###################################################
l_OvS = 380 ############################(length line plot OvS)############################################
gap_OvS_on = 900


for i in range(len(names)):
    f = open(names[i], mode = 'r', encoding='ISO-8859-15')
    print(names[i])
    lines = [line for line in f.readlines()]
    mV = np.array(lines[19:], dtype=float)
    t = np.arange(len(mV))
    f.close()
    
    name = names[i]
    name = name.strip('.dat_.txt')
    
    strain = lines[14]
    strain = strain.strip('strain')
    strain = strain.strip('=')
    strain = strain.strip('\n')


    comment2 = lines[17]
    comment2 = comment2.strip('comment2')
    comment2 = comment2.strip('=')
    comment2 = comment2.strip('\n')
    
    comment = ''

    plt.figure(figsize=(14, 7), dpi=100)
    plt.ylim(-24, 12)
    plt.ylabel('mV')
    plt.xlabel('ms')
    plt.suptitle('%s: %s' %(strain, comment2))
    plt.plot(mV)

# En esta sección del programa determinaremos el On, con base al momento del estímulo
    
    t_on = np.empty(0)
    peaks_on = np.empty(0)
    peaks_on, _ = find_peaks(mV[t_ini_on:], height= h_on)    
    B_on = False
    B_t_on = False
    if peaks_on.size > 0:
        for i_2 in range (len(peaks_on)):
            t_on = peaks_on[i_2] + t_ini_on
            counter_on = 0
            popt_on = np.empty(0)
            for i_3 in range (range_on):
                if peaks_on[i_2] + i_3 == len(t) - 1:
                    break
                if mV[peaks_on[i_2]+i_3] < 0:
                    counter_on = counter_on + 1
                if counter_on > limit_counter_on:
                    popt_on, pcov_on = curve_fit(func_pol_1, t[t_on:t_on+30], mV[t_on:t_on+30])
                    break
            if counter_on > limit_counter_on and popt_on.size > 0 and popt_on[0] < -0.04:            
                max_on = 50
                t_on = t_on + np.argmax(mV[t_on - max_on:t_on + max_on]) - max_on
                on = round(mV[t_on], 2)
                B_on = True
                B_t_on = True
                break               

    if not B_on:
        on = 0
        comment = 'Warning'                    
        for i_2_1 in range (len(t)):
            if len(mV[i_2_1:i_2_1+range_on]) == range_on:
                if np.all(mV[i_2_1:i_2_1+range_on] < 0):
                    popt_on_1, pcov_on_1 = curve_fit(func_pol_1, t[i_2_1:i_2_1+30], mV[i_2_1:i_2_1+30])
                    if popt_on_1.size > 0 and popt_on_1[0] < -0.04:
                        t_on = i_2_1
                        B_t_on = True
                        break 
    
    if B_t_on:
        plt.plot(t_on, on, color= 'orange',marker="x")
        plt.annotate('on = %.2f' %on, xy=(t_on - 400, on + 0.5))

## En esta sección del programa determinaremos el OvS y su area, en base al momento del estímulo
                   
    OvS = []
    area = []
    t_OvS_in = []
    t_OvS_end = []
    popt_OvS = np.empty(0)
    if B_t_on:
        t_OvS_in = t_on
        for i_7 in range (t_OvS_in, len(t)):
            if len(mV[i_7:i_7+20]) == 20:
                if np.all(mV[i_7:i_7+20] > 0):
                    t_OvS_in = i_7
                    break
        for i_8 in range (t_OvS_in, len(t)):              
            if len(mV[i_8:i_8+20]) == 20:
                if np.all(mV[i_8:i_8+20] < 0):
                    t_OvS_end = i_8
                    break          
        try:
            popt_OvS, pcov_OvS = curve_fit(func_pol_4, t[t_OvS_in:t_OvS_end], mV[t_OvS_in:t_OvS_end])
            y_OvS = func_pol_4(t[t_OvS_in:t_OvS_end], *popt_OvS)
            t_OvS = np.argmax(y_OvS) + t_OvS_in
            OvS = np.amax(y_OvS)
            OvS = round(OvS, 2)
            m_OvS = np.full((2*l_OvS, 1), OvS)
            m_t_OvS = np.arange(t_OvS-l_OvS, t_OvS+l_OvS)
            plt.plot(t[t_OvS_in:t_OvS_end], y_OvS, 'purple')
            plt.plot(m_t_OvS, m_OvS, color= 'red')
            plt.annotate('OvS = %.2f' %OvS, xy=(t_OvS-550, OvS+0.5))
            area = round(sum(mV[t_OvS_in:t_OvS_end]), 2)
            plt.plot(t[t_OvS_in:t_OvS_end], np.zeros_like(t[t_OvS_in:t_OvS_end]), color='black', linestyle="dashed")
            plt.annotate('Area = %.2f' %area, xy=(t_OvS-550, OvS/3))
        except (ValueError, RuntimeError, TypeError):
            OvS = np.nan      
            area = np.nan   
            comment = "Warning: OvS fit failed"        
    
## En esta sección del programa determinaremos el AmpR, con base al momento del estímulo

    AmpR = 0
    t_AmpR = 0
    if B_t_on:        
        if t_OvS_in:
            t_AmpR_in = t_on + int((t_OvS_in - t_on) / 2)
            ventana_t = t[t_on + 150 : t_AmpR_in]
            ventana_mV = mV[t_on + 150 : t_AmpR_in]
            
            if ventana_mV.size > 0:
                from scipy.signal import medfilt
                ventana_suave = medfilt(ventana_mV, kernel_size=21)
                idx_minimo = np.argmin(ventana_suave)                
                t_AmpR = ventana_t[idx_minimo]
                AmpR = ventana_suave[idx_minimo]
                AmpR = round(AmpR, 2)
                m_AmpR = np.full((2 * l_AmpR, 1), AmpR)
                m_t_AmpR = np.arange(t_AmpR - l_AmpR, t_AmpR + l_AmpR)                
                plt.plot(m_t_AmpR, m_AmpR, color='red')
                plt.annotate('AmpR = %.2f' %-AmpR, xy=(t_AmpR - 200, AmpR - 1))
                plt.plot(ventana_t, ventana_suave, color='purple')

        
# En esta sección del programa determinaremos el off, con base al momento del estímulo
         
    peaks_off = np.empty(0)
    t_off = np.empty(0)
    t_off_2 = np.empty(0)
    if B_t_on:
        t_off_in = t_OvS_in - 600
        t_off_end = t_OvS_in
        peaks_off, _ = find_peaks(-mV[t_off_in : t_off_end], height= h_off)
        if peaks_off.size > 0:
            for i_4 in range (len(peaks_off)):
                off = 0
                t_off = peaks_off[-(i_4+1)] + t_off_in
                try:
                    popt_off, pcov_off = curve_fit(func_pol_1, t[t_off:t_off+30], mV[t_off:t_off+30])
                except (RuntimeError, ValueError):
                    continue
                B_off = False
                B_off_2 = False
                if popt_off[0] > 0.012:                   
                    ventana_inicio = max(t_OvS_in - 350, t_off_in)
                    ventana_fin = min(t_OvS_in + 100, len(mV))
                    max_caida = -1
                    mejor_t_off_2 = t_off
                    mejor_t_off = t_off
                    for i_ms in range(ventana_inicio, ventana_fin - 10, 2):
                        for j_ms in range(i_ms + 4, min(i_ms + 80, ventana_fin), 2):
                            
                            if mV[i_ms] > mV[j_ms]:
                                caída_voltaje = mV[i_ms] - mV[j_ms]
                                
                                if caída_voltaje > max_caida:
                                    max_caida = caída_voltaje
                                    mejor_t_off_2 = i_ms
                                    mejor_t_off = j_ms
                    t_off_2 = mejor_t_off_2
                    t_off = mejor_t_off                  
                    B_off = True
                    B_off_2 = True
                    off = round((mV[t_off_2] - mV[t_off]), 2)                    
                    if off > min_off:
                        if len(peaks_off) >= 2 and (i_4 + 2) <= len(peaks_off):
                            if peaks_off[-(i_4+1)] - peaks_off[-(i_4+2)] > 2*range_off_mins:
                                break
                            if mV[peaks_off[-(i_4+1)] + t_off_in] < mV[peaks_off[-(i_4+2)] + t_off_in]:
                                break
                        else:
                            break 
    if B_off_2:
        plt.plot(t_off, mV[t_off], color= 'green', marker="x")
        plt.plot(t_off_2, mV[t_off_2], color= 'green', marker="x")
        plt.annotate('off = %.2f' %off, xy=(t_off-100, mV[t_off]-1))
    else:
        comment = 'Warning'                               

    try:
        resta_on_off = (t_off - t_on) if ('t_off' in locals() and 't_on' in locals() and isinstance(t_off, (int, float, np.integer)) and isinstance(t_on, (int, float, np.integer))) else np.nan
        resta_off_dur = (t_off - t_off_2) if ('t_off' in locals() and 't_off_2' in locals() and isinstance(t_off, (int, float, np.integer)) and isinstance(t_off_2, (int, float, np.integer))) else np.nan

        resultados_erg.append({
            'strain': strain if 'strain' in locals() else 'Desconocido',
            'condition': comment2 if 'comment2' in locals() else '', 
            'on': on if 'on' in locals() else np.nan,
            'off': off if 'off' in locals() else np.nan,
            'AmpR': -AmpR if 'AmpR' in locals() else np.nan,  
            'OvS': OvS if 'OvS' in locals() else np.nan,
            't_OvS_in': t_OvS_in if 't_OvS_in' in locals() else np.nan,
            't_OvS_end': t_OvS_end if 't_OvS_end' in locals() else np.nan,
            'area': area if 'area' in locals() else np.nan,
            't_on': t_on if 't_on' in locals() else np.nan,
            't_off': t_off if 't_off' in locals() else np.nan,
            't_off_2': t_off_2 if 't_off_2' in locals() else np.nan,
            't_on_off': resta_on_off,
            't_off_dur': resta_off_dur,
            'File': names[i],
            'comment': comment if 'comment' in locals() else ''
        })
    except Exception as e:
        print(f"Error al registrar datos del archivo {names[i]}: {e}")
   
    plt.savefig("%s/%s.png" %(output, name), dpi=100)
    plt.close()

## En esta sección del programa guardamos los parámetros

if resultados_erg:
    df_final = pd.DataFrame(resultados_erg)
    # output_dir = os.path.join(input_dir, "output")
    df_final.to_csv(os.path.join(output, "metrics_data.csv"), 
                    index=False, sep=',', decimal='.')
    
    print(f"\n=== ANÁLISIS TOTAL COMPLETADO ===")
    print(f"Se han analizado {len(df_final)} registros electrofisiológicos.")
    print(f"Archivo guardado en: {output}")
else:
    print("\n[ALERTA] No se recopilaron registros en la lista.")


ensayos th nuevos000440.dat_.txt
th-synf 40 días000624.dat_.txt
th-synf 40 días000559.dat_.txt
ensayos th nuevos000386.dat_.txt
th-synf 40 días000601.dat_.txt
ensayos th nuevos000431.dat_.txt
th-synf 40 días nicotina000642.dat_.txt
th-synf 40 días000631.dat_.txt
ensayos th nuevos000447.dat_.txt
th-synf 40 días nicotina000651.dat_.txt
ensayos th nuevos000450.dat_.txt
ensayos th nuevos000399.dat_.txt
ensayos th nuevos000435.dat_.txt
th-synf 40 días nicotina000646.dat_.txt
ensayos th nuevos000394.dat_.txt
ensayos th nuevos000426.dat_.txt
th-synf 40 días nicotina000641.dat_.txt
th-synf 40 días000572.dat_.txt
th-synf 40 días000550.dat_.txt
ensayos th nuevos000418.dat_.txt
ensayos th nuevos000405.dat_.txt
th-synf 40 días nicotina000647.dat_.txt
th-synf 40 días000620.dat_.txt
ensayos th nuevos000392.dat_.txt
th-synf 40 días000622.dat_.txt
th-synf 40 días nicotina000634.dat_.txt
th-synf 40 días nicotina000652.dat_.txt
th-synf 40 días000586.dat_.txt
th-synf 40 días000552.dat_.txt
ensayos th nue

En esta parte importamos nuestra tabla con las métricas obtenidas ("metrics_data.csv")
para evaluar el comportamiento de la población por genotipo y condición

In [21]:
df = df_final.sort_values(by=['strain', 'condition'], ignore_index=True)
grupos = df.groupby(['strain', 'condition'])

for (strain, condition), grupo in grupos:
    plt.figure(figsize=(14, 7), dpi=100)
    plt.ylim(-24, 12)
    plt.ylabel('mV')
    plt.xlabel('ms')
    plt.title(f'{strain}: {condition}')
    for idx, fila in grupo.iterrows():
        archivo = fila['File'] 
        
        if 't_on_off_2' in fila and fila['t_on_off_2'] < 0:
            continue   
        file_path = os.path.join(input_dir, archivo)        
        try:
            with open(file_path, mode='r', encoding='ISO-8859-15') as f:
                lines = f.readlines()
            mV = np.array(lines[19:], dtype=float)
            t_on = int(fila['t_on'])
            inicio = t_on - 400
            fin = t_on + 4000
            
            if fin > len(mV): fin = len(mV)
            if inicio < 0: inicio = 0
            
            mV_recortado = mV[inicio:fin]
            plt.plot(mV_recortado, label=archivo, color = 'black')            
        except Exception as e:
            print(f"No se pudo alinear el archivo {archivo}: {e}")            
    # plt.legend(loc='upper right', fontsize='xx-small', bbox_to_anchor=(1.15, 1))
    plt.tight_layout()
    
    strain_limpio = str(strain).strip().replace(' ', '_').replace('/', '_mas_').replace('>', '_')
    condition_limpia = str(condition).strip().replace(' ', '_').replace('/', '_').replace('.', '')    
    pop_fig = f"Fig_{strain_limpio}_{condition_limpia}.png"
    plt.savefig("%s/%s.png" %(output, pop_fig), dpi=100)
    plt.close()
    print(f"Gráfica grupal guardada con éxito: {pop_fig}")




Gráfica grupal guardada con éxito: Fig_TH-Synf_Moscas_de_40_días_alimentadas_con_cómida_con_nicotina_Driver_chileno.png
Gráfica grupal guardada con éxito: Fig_TH-Synf_Moscas_de_40_días_alimentadas_con_cómida_normal_Driver_chileno.png
Gráfica grupal guardada con éxito: Fig_TH_mas_+_Moscas_de_1_día_en_comida_normal_Driver_chileno_sin_expresar_synph.png
Gráfica grupal guardada con éxito: Fig_TH_Synf_Moscas_de_1_día_en_comida_normal__El_driver_de_TH_es_el_que_trajo_Angel_del_país_Chileno.png
Gráfica grupal guardada con éxito: Fig_TH_Synf_Moscas_de_1_día_en_comida_normal_Driver_chileno.png
Gráfica grupal guardada con éxito: Fig_TH_Synf_Moscas_de_1_día_en_comida_normal_Driver_chileno.png
Gráfica grupal guardada con éxito: Fig_TH_Synf_Moscas_de_40_días_alimentadas_con_cómida_normal_Driver_chileno.png
Gráfica grupal guardada con éxito: Fig_UAS-Synph_.png
Gráfica grupal guardada con éxito: Fig_o_Moscas_de_1_día_en_comida_normal_Driver_chileno_sin_expresar_synph.png


En esta parte del código generamos una gráfica promedio representativa de cada población

In [23]:
longitud_estandar = 4400 
diccionario_promedios = {
    'ms': np.arange(longitud_estandar)
}

for (strain, condition), grupo in grupos:
    senales_del_grupo = []
    for idx, fila in grupo.iterrows():
        archivo = fila['File']        
        if 't_on_off_2' in fila and fila['t_on_off_2'] < 0:
            continue          
        file_path = os.path.join(input_dir, archivo)
        try:
            with open(file_path, mode='r', encoding='ISO-8859-15') as f:
                lines = f.readlines()
            mV = np.array(lines[19:], dtype=float)
            t_on = int(fila['t_on'])
            inicio = t_on - 400
            fin = t_on + 4000
            if fin > len(mV): fin = len(mV)
            if inicio < 0: inicio = 0       
            mV_recortado = mV[inicio:fin]
            if len(mV_recortado) == longitud_estandar:
                senales_del_grupo.append(mV_recortado)
        except Exception as e:
            print(f"No se pudo procesar {archivo}: {e}")
            
    if len(senales_del_grupo) > 0:
        plt.figure(figsize=(14, 7), dpi=100)
        plt.ylim(-24, 12)
        plt.ylabel('mV')
        plt.xlabel('ms')
        plt.title(f'{strain}: {condition}')
        promedio_poblacion = np.mean(senales_del_grupo, axis=0)
        plt.plot(promedio_poblacion, color='black', linewidth=2.5, label='PROMEDIO')
        plt.legend(loc='upper right')
        plt.tight_layout()
        strain_limpio = str(strain).strip().replace(' ', '_').replace('/', '_').replace('>', '_').replace('<', '_')
        condition_limpia = str(condition).strip().replace(' ', '_').replace('/', '_').replace('.', '').replace(',', '')

        pop_avg_fig = f"Fig_Promedio_{strain_limpio}_{condition_limpia}.png"
        plt.savefig("%s/%s.png" %(output, pop_avg_fig), dpi=100)
        plt.close()
        
        nombre_columna = f"{strain_limpio}__{condition_limpia}"
        diccionario_promedios[nombre_columna] = promedio_poblacion
        
        print(f"Promedio de {nombre_columna} acumulado en el DataFrame global.")
    else:
        print(f"No hay archivos suficientes para generar el promedio de {strain}: {condition}")

if len(diccionario_promedios) > 1:
    df_global_average = pd.DataFrame(diccionario_promedios)
    df_global_average.to_csv(os.path.join(output, "global_average.csv"), 
                    index=False, sep=',', decimal='.')
    print(f"\n=== PROCESAMIENTO GLOBAL COMPLETADO ===")
    print(f"Se han unificado todas las poblaciones en un único DataFrame.")
    print(f"Columnas creadas: {list(df_global_average.columns)}")
    print(f"Archivo unificado guardado con éxito en: {output}")


Promedio de TH-Synf__Moscas_de_40_días_alimentadas_con_cómida_con_nicotina_Driver_chileno acumulado en el DataFrame global.
Promedio de TH-Synf__Moscas_de_40_días_alimentadas_con_cómida_normal_Driver_chileno acumulado en el DataFrame global.
Promedio de TH_+__Moscas_de_1_día_en_comida_normal_Driver_chileno_sin_expresar_synph acumulado en el DataFrame global.
Promedio de TH_Synf__Moscas_de_1_día_en_comida_normal__El_driver_de_TH_es_el_que_trajo_Angel_del_país_Chileno acumulado en el DataFrame global.
Promedio de TH_Synf__Moscas_de_1_día_en_comida_normal_Driver_chileno acumulado en el DataFrame global.
Promedio de TH_Synf__Moscas_de_1_día_en_comida_normal_Driver_chileno acumulado en el DataFrame global.
Promedio de TH_Synf__Moscas_de_40_días_alimentadas_con_cómida_normal_Driver_chileno acumulado en el DataFrame global.
Promedio de UAS-Synph__ acumulado en el DataFrame global.
Promedio de o__Moscas_de_1_día_en_comida_normal_Driver_chileno_sin_expresar_synph acumulado en el DataFrame globa